## Data Transformation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_parquet('../data/raw/transactions.parquet')

In [3]:
df.shape

(6362620, 11)

In [5]:
# create a scoped dataset with only TRANSFER and CASH_OUT transactions
df_scope = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
df_scope.shape
df_scope['type'].value_counts()

type
CASH_OUT    2237500
TRANSFER     532909
Name: count, dtype: int64

In [22]:
df_scope.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,dest_is_merchant
2,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0,0
15,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0,0
19,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0,0
24,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0,0


In [6]:
df_scope.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [7]:
df_scope.duplicated().sum()

0

In [8]:
numeric_cols = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest'
]

for col in numeric_cols:
    print(f"{col}: {(df_scope[col] < 0).sum()} negative values")

amount: 0 negative values
oldbalanceOrg: 0 negative values
newbalanceOrig: 0 negative values
oldbalanceDest: 0 negative values
newbalanceDest: 0 negative values


In [31]:
print("Zero amount rows:", (df_scope['amount'] == 0).sum())

Zero amount rows: 16


In [32]:
zero_amount_rows = df_scope[df_scope['amount'] == 0]
print(zero_amount_rows)

         step      type  amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
2736447   212  CASH_OUT     0.0  C1510987794            0.0             0.0   
3247298   250  CASH_OUT     0.0   C521393327            0.0             0.0   
3760289   279  CASH_OUT     0.0   C539112012            0.0             0.0   
5563714   387  CASH_OUT     0.0  C1294472700            0.0             0.0   
5996408   425  CASH_OUT     0.0   C832555372            0.0             0.0   
5996410   425  CASH_OUT     0.0    C69493310            0.0             0.0   
6168500   554  CASH_OUT     0.0    C10965156            0.0             0.0   
6205440   586  CASH_OUT     0.0  C1303719003            0.0             0.0   
6266414   617  CASH_OUT     0.0  C1971175979            0.0             0.0   
6281483   646  CASH_OUT     0.0  C2060908932            0.0             0.0   
6281485   646  CASH_OUT     0.0  C1997645312            0.0             0.0   
6296015   671  CASH_OUT     0.0  C1960007029        

In [ ]:
df_scope['amount'].describe(
    percentiles=[0.50, 0.75, 0.90, 0.99, 0.999] ## right skewed distribution, by mean and std
)

count    2.770409e+06
mean     3.175361e+05
std      8.877897e+05
min      0.000000e+00
50%      1.712609e+05
75%      3.067912e+05
90%      5.448676e+05
99%      2.650036e+06
99.9%    1.000000e+07
max      9.244552e+07
Name: amount, dtype: float64

In [ ]:
df_scope['dest_is_merchant'] = (
    df_scope['nameDest'].str.startswith('M')
    .astype(int)
)

In [24]:
print(df_scope[['nameDest', 'dest_is_merchant']].head(10))

       nameDest  dest_is_merchant
2    C553264065                 0
3     C38997010                 0
15   C476402209                 0
19  C1100439041                 0
24   C932583850                 0
42  C1509514333                 0
47   C824009085                 0
48   C248609774                 0
51  C2001112025                 0
58  C1937962514                 0


In [23]:
print(
    df_scope['dest_is_merchant']
    .map({0: 'Customer', 1: 'Merchant'})
    .value_counts()
)

dest_is_merchant
Customer    2770409
Name: count, dtype: int64


In [18]:
cols_to_drop_later = [
    'nameOrig',
    'nameDest',
    'isFlaggedFraud'
]

In [19]:
print(cols_to_drop_later)

['nameOrig', 'nameDest', 'isFlaggedFraud']


In [ ]:
df_scope['type'] = df_scope['type'].astype('category') # category is pandas specific 

print(df_scope.dtypes)

step                   int64
type                category
amount               float64
nameOrig              object
oldbalanceOrg        float64
newbalanceOrig       float64
nameDest              object
oldbalanceDest       float64
newbalanceDest       float64
isFraud                int64
isFlaggedFraud         int64
dest_is_merchant       int64
dtype: object


In [34]:
df_scope['isFraud'] = df_scope['isFraud'].astype('int8')
df_scope['isFlaggedFraud'] = df_scope['isFlaggedFraud'].astype('int8')
df_scope['dest_is_merchant'] = df_scope['dest_is_merchant'].astype('int8')
df_scope.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2770409 entries, 2 to 6362619
Data columns (total 12 columns):
 #   Column            Dtype   
---  ------            -----   
 0   step              int64   
 1   type              category
 2   amount            float64 
 3   nameOrig          object  
 4   oldbalanceOrg     float64 
 5   newbalanceOrig    float64 
 6   nameDest          object  
 7   oldbalanceDest    float64 
 8   newbalanceDest    float64 
 9   isFraud           int8    
 10  isFlaggedFraud    int8    
 11  dest_is_merchant  int8    
dtypes: category(1), float64(5), int64(1), int8(3), object(2)
memory usage: 200.8+ MB


In [27]:
df_scope.to_parquet(
    '../data/processed/paysim_clean.parquet',
    index=False
)

In [28]:
df_cleaned = pd.read_parquet('../data/processed/paysim_clean.parquet')

In [29]:
df_cleaned.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,dest_is_merchant
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0,0
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0,0
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0,0
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0,0
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0,0


In [30]:
# Overall fraud rate
fraud_rate = df_cleaned['isFraud'].mean() * 100

print(f"Fraud rate: {fraud_rate:.4f}%")

Fraud rate: 0.2965%


In [35]:
check = pd.read_parquet("../data/processed/paysim_clean.parquet")
print("rows:", f"{len(check):,}")
print("cols:", list(check.columns))
print("types present:", check["type"].unique().tolist())
print("fraud rate:", f"{check['isFraud'].mean()*100:.4f}%")
print("merchant flag added:", "dest_is_merchant" in check.columns)

rows: 2,770,409
cols: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'dest_is_merchant']
types present: ['TRANSFER', 'CASH_OUT']
fraud rate: 0.2965%
merchant flag added: True
